# Score-CAM

Grad-CAM, Grad-CAM++, Score-CAM and Faster-Score-CAM with Keras 3 / `tf.keras`.

The implementation lives in the [`scorecam`](./scorecam) package; this notebook
only calls it. Outputs are stripped before commit — run the cells to see the
figures, or look at [`result/`](./result) for the rendered versions.

```bash
uv sync --group dev
uv run jupyter lab
```

## Procedure of Score-CAM

1. upsample the output of the final conv layer to the original input size
2. normalize the raw activation value in each activation map into `[0, 1]`
3. project the highlighted area in the activation map to the original input space by multiplying the normalized activation map
4. feed the masked inputs into the CNN and softmax
5. define the weight as the score of the target class
6. get the final class-discriminative localization map as a linear weighted combination of all activation maps

Steps 1-6 are implemented in [`scorecam/score_cam.py`](./scorecam/score_cam.py),
which also documents the two places where the paper and the authors' released
code disagree. Every CAM function returns a 2-D `float32` array in `[0, 1]`.

## Score-CAM with VGG16

In [ ]:
import os

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import matplotlib.pyplot as plt
import numpy as np
from keras.applications.vgg16 import VGG16, decode_predictions, preprocess_input
from keras.utils import load_img

from scorecam import ScoreCam, read_and_preprocess_img, read_img, superimpose

model = VGG16(include_top=True, weights="imagenet")
layer_name = "block5_conv3"

img_path = "./image/hummingbird.jpg"
img_array = read_and_preprocess_img(img_path, size=(224, 224))

top = decode_predictions(model.predict(img_array, verbose=0), top=3)[0]
print("\n".join(f"{name:>16s} {p:.4f}" for _, name, p in top))

score_cam = ScoreCam(model, img_array, layer_name)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(np.array(load_img(img_path), dtype=np.uint8)); ax[0].set_title("input")
ax[1].imshow(score_cam);                                    ax[1].set_title("Score-CAM (14x14)")
ax[2].imshow(superimpose(img_path, score_cam));             ax[2].set_title("superimposed")
for a in ax:
    a.axis("off")
plt.show()

## Explaining a class other than the top-1

Every CAM takes `class_index=`. ImageNet class 285 is `Egyptian_cat`, 207 is `golden_retriever`.

In [ ]:
cat_dog = "./image/cat_dog.png"
cat_dog_array = read_and_preprocess_img(cat_dog, size=(224, 224))

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(np.array(load_img(cat_dog), dtype=np.uint8)); ax[0].set_title("input")
for a, (cls, label) in zip(ax[1:], [(285, "Egyptian_cat"), (207, "golden_retriever")]):
    cam = ScoreCam(model, cat_dog_array, layer_name, class_index=cls, max_N=10)
    a.imshow(superimpose(cat_dog, cam))
    a.set_title(f"Faster-Score-CAM for {label} ({cls})")
for a in ax:
    a.axis("off")
plt.show()

## Compare Grad-CAM, Grad-CAM++, Score-CAM and Faster-Score-CAM

In [ ]:
from scorecam import build_guided_model
from scorecam.plotting import comparison_figure, explain

guided_model = build_guided_model(model)

maps = explain(model, img_path, layer_name, preprocess_input, guided_model=guided_model)
comparison_figure(img_path, maps, rows=("overlay", "guided"))
plt.show()

## Processing time

`max_N` is what makes Faster-Score-CAM fast: it keeps only the `max_N`
highest-variance activation maps, so the number of forward passes drops from
512 (VGG16's `block5_conv3`) to `max_N`.

In [ ]:
from scorecam import GradCam, GradCamPlusPlus, GuidedBackPropagation

print("Grad-CAM")
%timeit -n1 -r1 GradCam(model, img_array, layer_name)
print("Grad-CAM++")
%timeit -n1 -r1 GradCamPlusPlus(model, img_array, layer_name)
print("Score-CAM")
%timeit -n1 -r1 ScoreCam(model, img_array, layer_name)
print("Faster-Score-CAM N=10")
%timeit -n1 -r1 ScoreCam(model, img_array, layer_name, max_N=10)
print("Faster-Score-CAM N=3")
%timeit -n1 -r1 ScoreCam(model, img_array, layer_name, max_N=3)
print("Guided-BP")
%timeit -n1 -r1 GuidedBackPropagation(guided_model, img_array, layer_name)

In [ ]:
# How much does dropping channels cost in fidelity?
full = ScoreCam(model, img_array, layer_name)
for n in (100, 30, 10, 3, 1):
    fast = ScoreCam(model, img_array, layer_name, max_N=n)
    r = np.corrcoef(full.ravel(), fast.ravel())[0, 1]
    print(f"max_N={n:4d}  pearson r vs full Score-CAM = {r:+.4f}")

## The paper and the authors' code disagree

Algorithm 1 of the [Score-CAM paper](https://arxiv.org/abs/1910.01279) takes the
softmax of the target-class **logits across channels**. The authors' released
PyTorch code takes the softmax across **classes** and reads off the target
probability. This repository defaults to the latter, because Algorithm 1 read
literally is close to degenerate — exponentiating raw logits concentrates almost
all the weight on a couple of channels.

In [ ]:
from scorecam.diagnostics import weight_concentration

raw = read_img(img_path, size=(224, 224))

# Measured under ScoreCam's own defaults, so these numbers describe what
# ScoreCam(...) actually computes. Add raw_img_array=/preprocess_fn= to measure
# under the paper's raw-pixel masking instead; the conclusion is the same.
stats = weight_concentration(model, img_array, layer_name)

print(f"target-class logits over {stats['n_channels']} masked inputs: "
      f"min={stats['logit_min']:.2f} max={stats['logit_max']:.2f}")
print(f"{'weight_mode':<12}{'largest':>10}{'top-10':>10}{'exp(H)':>10}")
for mode in ("paper", "reference"):
    s = stats[mode]
    print(f"{mode:<12}{s['largest_weight']:>9.1%}{s['top10_share']:>10.1%}"
          f"{s['effective_channels']:>10.1f}")

In [ ]:
# ... and what that looks like
variants = {
    "reference (default)": dict(),
    "paper: channel softmax": dict(weight_mode="paper"),
    "paper: + raw-pixel mask": dict(weight_mode="paper", raw_img_array=raw,
                                    preprocess_fn=preprocess_input),
}
fig, ax = plt.subplots(1, len(variants) + 1, figsize=(5 * (len(variants) + 1), 5))
ax[0].imshow(np.array(load_img(img_path), dtype=np.uint8)); ax[0].set_title("input")
for a, (label, kwargs) in zip(ax[1:], variants.items()):
    a.imshow(superimpose(img_path, ScoreCam(model, img_array, layer_name, **kwargs)))
    a.set_title(label)
for a in ax:
    a.axis("off")
plt.show()

## Visualization for other images

In [ ]:
for basename in ["cat_dog.png", "collies.JPG", "water-bird.JPEG"]:
    path = f"./image/{basename}"
    label = decode_predictions(
        model.predict(read_and_preprocess_img(path), verbose=0), top=1
    )[0][0][1]
    maps = explain(model, path, layer_name, preprocess_input, guided_model=guided_model)
    comparison_figure(path, maps, title=f"{basename} -- {label}")
    plt.show()

# The committed figures in result/ come from the same code:
#   uv run python scripts/regenerate_results.py

# Applying Score-CAM to your own model

## [DAGM dataset](https://resources.mpi-inf.mpg.de/conference/dagm/2007/prizes.html)

Six classes of synthetic textures with weakly-labelled defects. Roughly 2 GB.

In [ ]:
!mkdir -p DAGM
!for i in {1..6}; do wget -q -P DAGM "https://conferences.mpi-inf.mpg.de/dagm/2007/Class${i}.zip";     unzip -d DAGM -q -o "DAGM/Class${i}.zip"; done
!for i in {1..6}; do wget -q -P DAGM "https://conferences.mpi-inf.mpg.de/dagm/2007/Class${i}_def.zip"; unzip -d DAGM -q -o "DAGM/Class${i}_def.zip"; done

In [ ]:
import glob

from keras.utils import to_categorical
from sklearn.model_selection import train_test_split

from keras.applications.resnet50 import preprocess_input as resnet_preprocess_input

num_classes = 2
img_size = (224, 224)
dagm_path = "DAGM"


def get_dagm_data(names):
    x, y = [], []
    for i, name in enumerate(names):
        for path in sorted(glob.glob(f"{dagm_path}/{name}/*.png")):
            # NOTE: the ResNet below needs ResNet preprocessing. Before v0.2 this
            # silently used VGG16's, because read_and_preprocess_img hard-coded it.
            x.append(read_and_preprocess_img(path, size=img_size,
                                             preprocess_fn=resnet_preprocess_input))
            y.append(i)

    x = np.concatenate(x, axis=0)
    y = np.array(y)

    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=0.2, random_state=111
    )
    print(x_train.shape[0], "train samples")
    print(x_test.shape[0], "test samples")
    return (x_train, x_test,
            to_categorical(y_train, num_classes), to_categorical(y_test, num_classes))


x_train, x_test, y_train, y_test = get_dagm_data(["Class1", "Class1_def"])

## ResNet

The backbone is cut at `conv3_block4_add`, whose 28x28 feature map is fine
enough to localize the small DAGM defects — the full 7x7 output of ResNet50 is
not. Pre-v0.2 this was written as `model.layers[-98]`, which means whatever
Keras happens to index there.

In [ ]:
from keras.applications.resnet50 import ResNet50
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.layers import Activation, Dense, GlobalAveragePooling2D, Input
from keras.models import Model
from keras.optimizers import Adam

TRUNCATE_AT = "conv3_block4_add"


def build_ResNet():
    backbone = ResNet50(include_top=True, input_tensor=Input(shape=(*img_size, 3)))

    x = backbone.get_layer(TRUNCATE_AT).output
    x = Activation("relu", name="act_last")(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense(2, name="dense_out")(x)
    outputs = Activation("softmax")(x)

    model = Model(backbone.input, outputs)
    model.compile(loss="binary_crossentropy",
                  optimizer=Adam(amsgrad=True),
                  metrics=["accuracy"])
    return model


dagm_model = build_ResNet()
checkpoint = "./resnet_weight_DAGM.weights.h5"

history = dagm_model.fit(
    x_train, y_train,
    batch_size=32,
    epochs=30,
    validation_data=(x_test, y_test),
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=15, verbose=1, mode="auto"),
        ModelCheckpoint(filepath=checkpoint, monitor="val_loss", verbose=1,
                        save_best_only=True, save_weights_only=True, mode="auto"),
    ],
    class_weight={0: 1.0, 1: 6.0},
    shuffle=True,
)
dagm_model.load_weights(checkpoint)

### Guided Backpropagation on your own model

`build_guided_model` takes the built model directly now; it no longer needs a
factory function.

In [ ]:
dagm_layer_name = "act_last"
dagm_img_path = f"{dagm_path}/Class1_def/12.png"

dagm_guided_model = build_guided_model(dagm_model)

maps = explain(dagm_model, dagm_img_path, dagm_layer_name, resnet_preprocess_input,
               guided_model=dagm_guided_model)

comparison_figure(dagm_img_path, maps, rows=("overlay",))
plt.show()
comparison_figure(dagm_img_path, maps, rows=("overlay",), emphasize=True)
plt.show()
comparison_figure(dagm_img_path, maps, rows=("guided",))
plt.show()

## DAGM Class1 - Class6

`emphasize=True` pushes the heatmap through a steep sigmoid, which is what makes
these low-contrast texture defects visible.

In [ ]:
from scorecam.plotting import METHOD_LABELS  # noqa: F401


def dagm_figure(model, class_name, out_path, n=5):
    fig = plt.figure(figsize=(20, 8))
    paths = sorted(glob.glob(f"{dagm_path}/{class_name}_def/*.png"))[:n]
    for i, path in enumerate(paths):
        img_array = read_and_preprocess_img(path, size=img_size,
                                            preprocess_fn=resnet_preprocess_input)
        cam = ScoreCam(model, img_array, "act_last", max_N=10)

        ax = fig.add_subplot(2, n, i + 1)
        ax.imshow(np.array(load_img(path), dtype=np.uint8))
        ax.set_title(os.path.basename(path)); ax.axis("off")

        ax = fig.add_subplot(2, n, n + i + 1)
        ax.imshow(superimpose(path, cam, emphasize=True))
        ax.set_title(os.path.basename(path)); ax.axis("off")
    fig.tight_layout()
    fig.savefig(out_path)
    return fig


dagm_figure(dagm_model, "Class1", "./result/Class1_result_0.png")
plt.show()

In [ ]:
for class_name in ["Class2", "Class3", "Class4", "Class5", "Class6"]:
    print(f"{class_name} {'=' * 60}")
    x_train, x_test, y_train, y_test = get_dagm_data([class_name, f"{class_name}_def"])

    model_i = build_ResNet()
    checkpoint_i = f"./resnet_weight_DAGM_{class_name}.weights.h5"
    model_i.fit(
        x_train, y_train,
        batch_size=32,
        epochs=30,
        validation_data=(x_test, y_test),
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=15, verbose=1, mode="auto"),
            ModelCheckpoint(filepath=checkpoint_i, monitor="val_loss", verbose=1,
                            save_best_only=True, save_weights_only=True, mode="auto"),
        ],
        class_weight={0: 1.0, 1: 6.0},
        shuffle=True,
    )
    model_i.load_weights(checkpoint_i)

    dagm_figure(model_i, class_name, f"./result/{class_name}_result_0.png")
    plt.show()